In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -r '/content/drive/MyDrive/requirement/requirements.txt'

In [2]:
import os

# DeepFillv2 セットアップ
HOME = "/content"
%cd {HOME}
!git clone https://github.com/vrindaprabhu/deepfillv2_colab.git
!gdown "https://drive.google.com/u/0/uc?id=1uMghKl883-9hDLhSiI8lRbHCzCmmRwV-&export=download"
!mv /content/deepfillv2_WGAN_G_epoch40_batchsize4.pth deepfillv2_colab/model/deepfillv2_WGAN.pth

# GroundingDINO セットアップ
%cd {HOME}
!git clone https://github.com/IDEA-Research/GroundingDINO.git
%cd {HOME}/GroundingDINO
!pip install -q -e .
!pip install -q roboflow

# GroundingDINOのコンフィグとウェイトのダウンロード
CONFIG_PATH = os.path.join(HOME, "GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py")
print(CONFIG_PATH, "; exist:", os.path.isfile(CONFIG_PATH))

!mkdir -p {HOME}/weights
%cd {HOME}/weights
!wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
WEIGHTS_NAME = "groundingdino_swint_ogc.pth"
WEIGHTS_PATH = os.path.join(HOME, "weights", WEIGHTS_NAME)
print(WEIGHTS_PATH, "; exist:", os.path.isfile(WEIGHTS_PATH))
%cd {HOME}/GroundingDINO

/content
fatal: destination path 'deepfillv2_colab' already exists and is not an empty directory.
Downloading...
From: https://drive.google.com/u/0/uc?id=1uMghKl883-9hDLhSiI8lRbHCzCmmRwV-&export=download
To: /content/deepfillv2_WGAN_G_epoch40_batchsize4.pth
100% 64.8M/64.8M [00:00<00:00, 261MB/s]
/content
fatal: destination path 'GroundingDINO' already exists and is not an empty directory.
/content/GroundingDINO
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.7/254.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.9/80.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.8 MB/s eta 0:00:00
/content/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py ; exist: True
/content/weights
/content/weights/groundingdino_swint_ogc.pth

In [3]:
# 認証トークンをファイルから読み込み
with open('/content/drive/MyDrive/ngrok_token.txt', 'r') as file:
    ngrok_auth_token = file.read().strip()

# 認証トークンを使用して ngrok に設定
!ngrok authtoken {ngrok_auth_token}


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
import os
import supervision as sv
from PIL import Image, ImageFilter
import numpy as np
import cv2
import pycocotools.mask as mask_util
from groundingdino.util.inference import load_model, load_image, predict, annotate
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, HTMLResponse
import shutil
import json
from pathlib import Path
import nest_asyncio
import uvicorn
from pyngrok import ngrok
from diffusers import StableDiffusionInpaintPipeline
import torch
from simple_lama_inpainting import SimpleLama
from sklearn.cluster import (
    KMeans, AgglomerativeClustering, DBSCAN, MiniBatchKMeans, Birch,
    SpectralClustering, MeanShift, OPTICS
)
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier
from torchvision import transforms
import threading
import concurrent.futures
from typing import Tuple
from types import SimpleNamespace
import subprocess
import uuid
from datetime import datetime
from ultralytics import YOLO
import math

HOME = "/content"
CONFIG_PATH = os.path.join(HOME, "GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py")
WEIGHTS_NAME = "groundingdino_swint_ogc.pth"
WEIGHTS_PATH = os.path.join(HOME, "weights", WEIGHTS_NAME)
detection_model = load_model(CONFIG_PATH, WEIGHTS_PATH)

dangerarray=[10,30,90,50,80,20,40,70,100,60]#ここに各クラスターの危険度を設定しておく
#ここで認識する精度を上げたり下げたりできる

thresholds = {
    'text': 0.1,
    'name tag': 0.1,
    'license plate': 0.3,
    'Mail': 0.3,
    'Documents': 0.3,
    'QR codes': 0.4,
    'barcodes': 0.4,
    'map': 0.5,
    'digital screens': 0.6,
    'information board': 0.5,
    'signboard': 0.3,
    'poster': 0.8,
    'sign': 0.3,
    'logo': 0.3,
    'card': 0.4,
    'window': 0.2,
    'mirror': 0.2,
    'Famous landmark': 0.7,
    'cardboard': 0.6,
    'manhole': 0.6,
    'utility pole': 0.7
}

'''
#下は反比例
# 閾値と下限の設定
thresholds = {
    'text': 5,
    'Name tag':11,
    'License plate': 9,
    'Mail': 60,
    'Documents': 60,
    'QR codes': 60,
    'barcodes': 60,
    'Map': 50,
    'Digital screens': 40,
    'information board': 40,
    'signboard': 30,
    'poster': 22,
    'sign': 20,
    'logo': 30,
    'card': 60,
    'window': 60,
    'mirror': 40,
    'Famous landmark': 80,
    'cardboard': 60,
    'manhole': 20,
    'utility pole': 9
}

lower_limits = {
    'text': 0.1,
    'Name tag': 0.2,
    'License plate': 0.2,
    'Mail': 0.3,
    'Documents': 0.1,
    'QR codes': 0.4,
    'barcodes': 0.4,
    'Map': 0.6,
    'Digital screens': 0.3,
    'information board': 0.7,
    'signboard': 0.2,
    'poster': 0.5,
    'sign': 0.2,
    'logo': 0.2,
    'card': 0.6,
    'window': 0.7,
    'mirror': 0.5,
    'Famous landmark': 0.8,
    'cardboard': 0.7,
    'manhole': 0.6,
    'utility pole': 0.3
}
'''

# Define paths
HOME = "/content"
CONFIG_PATH = os.path.join(HOME, "GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py")
WEIGHTS_NAME = "groundingdino_swint_ogc.pth"
WEIGHTS_PATH = os.path.join(HOME, "weights", WEIGHTS_NAME)
from PIL import Image
# process_image_norm関数内でimage_pathを使用
def process_image_norm(image_path):
        detection_model = load_model(CONFIG_PATH, WEIGHTS_PATH)

        image_source, image = load_image(image_path)
        object_vector = np.zeros(len(TEXT_PROMPTS))

        for i, text_prompt in enumerate(TEXT_PROMPTS):
            boxes, logits, phrases = predict(
                model=detection_model,
                image=image,
                caption=text_prompt,
                box_threshold=BOX_THRESHOLD,
                text_threshold=TEXT_THRESHOLD
            )

            if len(logits) > 0:
                logits_np = np.array(logits)
                object_vector[i] = np.sum(logits_np)
        magnitude = np.linalg.norm(object_vector)
        return magnitude

def is_bright(pixel):
    # ピクセルの輝度を計算して明るさを判定する
    r, g, b = pixel
    brightness = (0.299 * r + 0.587 * g + 0.114 * b)  # 輝度の計算
    return brightness > 127  # 閾値を127に設定

def analyze_mask_brightness(original_image_path, mask_image_path):
    # 画像を開く
    original_img = Image.open(original_image_path).convert('RGB')
    mask_img = Image.open(mask_image_path).convert('L')  # グレースケールに変換

    width, height = original_img.size

    if mask_img.size != (width, height):
        print("エラー: マスク画像と元画像のサイズが一致していません。")
        return

    # 明るいピクセルと暗いピクセルのカウント
    bright_count = 0
    dark_count = 0

    for y in range(height):
        for x in range(width):
            mask_value = mask_img.getpixel((x, y))
            if mask_value > 127:  # マスクが白（対象領域）ならば
                pixel = original_img.getpixel((x, y))
                if is_bright(pixel):
                    bright_count += 1
                else:
                    dark_count += 1

    # 明るさの結果を判定
    brightness_result = 1 if bright_count > dark_count else 2

    return brightness_result

def classify_mask_size(mask_image_path, small_threshold, medium_threshold, large_threshold):
    # マスク画像を開く
    mask_img = Image.open(mask_image_path).convert('L')  # グレースケールに変換

    width, height = mask_img.size
    total_pixels = width * height
    white_pixel_count = 0

    # マスク画像の白いピクセルをカウント
    for y in range(height):
        for x in range(width):
            mask_value = mask_img.getpixel((x, y))
            if mask_value > 127:  # 白いピクセルと判断
                white_pixel_count += 1

    # 白いピクセルの割合を計算
    mask_area_ratio = (white_pixel_count / total_pixels) * 100

    # マスクサイズを分類
    if mask_area_ratio <= small_threshold:
        size_category = 1  # すごく小さい
    elif mask_area_ratio <= medium_threshold:
        size_category = 2  # 小さい
    elif mask_area_ratio <= large_threshold:
        size_category = 3  # 大きい
    else:
        size_category = 4  # すごく大きい

    return size_category

def analyze_mask_combined(original_image_path, mask_image_path, small_threshold, medium_threshold, large_threshold):
    # マスクの大きさを判定
    size_category = classify_mask_size(mask_image_path, small_threshold, medium_threshold, large_threshold)

    # マスク部分の明るさを判定
    brightness_result = analyze_mask_brightness(original_image_path, mask_image_path)

    # 結果を出力
    size_text = {1: "すごく小さい", 2: "小さい", 3: "大きい", 4: "すごく大きい"}
    print(f"マスクの大きさ: {size_text[size_category]} ({size_category})")
    print(f"マスクの明るさ: {brightness_result}")
    result={
            'size':size_category,
            'brightness':brightness_result
            }
    return result


def quantize_model(model):
    # モデルを動的量子化
    model_quantized = torch.quantization.quantize_dynamic(
        model, {torch.nn.Linear}, dtype=torch.qint8
    )
    return model_quantized
'''
thresholds = {
    'text': 0.1,
    #'Name tag': 0.1,
    'License plate': 0.3,
    #'Mail': 0.3,
    'Documents': 0.3,
    #'QR codes': 0.4,
    #'barcodes': 0.4,
    #'Map': 0.5,
    'Digital screens': 0.6,
    #'information board': 0.5,
    'signboard': 0.3,
    'poster': 0.8,
    'sign': 0.3,
    #'logo': 0.3,
    #'card': 0.4,
    #'window': 0.2,
    #'mirror': 0.2,
    #'Famous landmark': 0.7,
    #'cardboard': 0.6,
    #'manhole': 0.6,
    'utility pole': 0.7
}
'''
#この下で消去対象を決定
def decide_to_object(risk_level):
    '''
    tex = [
        'text','Name tag', 'License plate', 'Mail', 'Documents', 'QR codes',
        'barcodes', 'Map', 'Digital screens', 'information board',
        'signboard', 'poster', 'sign', 'utility pole'

    ]
    '''
    tex = [
        'text', 'License plate', 'Digital screens',
        'signboard', 'poster', 'sign', 'logo', 'card', 'window', 'mirror',
        'Famous landmark', 'cardboard', 'manhole', 'utility pole'

    ]


    #この配列の要素の順番を変えると消える順番が変わる。
    risk_level = int(risk_level / 20)*(len(tex)/10)#個数決定(1/2)
    return tex[:int(risk_level)+1]

def create_mask(image, x1, y1, x2, y2):
    # Create a black image with the same size as the input image
    mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)

    # Draw a white rectangle on the mask where the object is located
    cv2.rectangle(mask, (int(x1), int(y1)), (int(x2), int(y2)), 255, -1)

    return mask

import numpy as np
import matplotlib.pyplot as plt

#この下のコードは特定の領域をマスクしないタイプのコード
import uuid
from datetime import datetime

def special_process_image(risk_level, image_path, point1, point2, thresholds=None):
    if thresholds is None:
        thresholds = {}

    # 現在のタイムスタンプを生成
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # 一意な識別子を生成
    unique_id = uuid.uuid4().hex

    # ファイル名を生成
    debug_image_filename = f"/content/debug_image_with_rectangle_{timestamp}_{unique_id}.jpg"
    mask_filename = f"/content/final_mask_{timestamp}_{unique_id}.jpg"

    # リスクレベルに基づいてしきい値を調整する関数
    def logistic_decay(risk_level, k=0.1, r0=50):
        return 1 / (1 + np.exp(-k * (risk_level - r0)))

    threshold_results = {key: [] for key in thresholds.keys()}
    decay_factor = logistic_decay(risk_level)

    # リスクレベルに応じたしきい値の調整
    adjusted_thresholds = {key: max(value - decay_factor + 0.8, 0.01) / 2 for key, value in thresholds.items()}
    detection_model = load_model(CONFIG_PATH, WEIGHTS_PATH)
    IMAGE_PATH = image_path
    LABELS = decide_to_object(risk_level)
    image_source, image = load_image(IMAGE_PATH)

    all_boxes = []
    all_logits = []

    for label in LABELS:
        boxes, logits, _ = predict(
            model=detection_model,
            image=image,
            caption=label,
            box_threshold=0.3,
            text_threshold=0.3
        )
        all_boxes.extend(boxes)
        all_logits.extend(logits)

    # PyTorch テンソルを NumPy 配列に変換し、uint8 に変換
    image_np = image.permute(1, 2, 0).cpu().numpy()
    image_np = np.clip(image_np * 255, 0, 255).astype(np.uint8)

    # デバッグ用の画像を作成
    debug_image = image_np.copy()

    # マスクを黒で初期化
    mask = np.zeros((image_np.shape[0], image_np.shape[1]), dtype=np.uint8)

    for i, box in enumerate(all_boxes):
        x, y, w, h = box
        confidence = all_logits[i]
        object_type = LABELS[i % len(LABELS)]

        # オブジェクトタイプに応じたしきい値を取得、デフォルトは0.5
        threshold = adjusted_thresholds.get(object_type, 0.5)
        print(threshold)
        if confidence >= threshold:
            x1 = int((x - w / 2) * image_np.shape[1])
            y1 = int((y - h / 2) * image_np.shape[0])
            x2 = int((x + w / 2) * image_np.shape[1])
            y2 = int((y + h / 2) * image_np.shape[0])

            # 座標が画像の範囲内に収まるように修正
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(image_np.shape[1], x2), min(image_np.shape[0], y2)

            mask = create_mask(image_np, x1, y1, x2, y2) | mask

    # 点を絶対座標に変換
    p1_x, p1_y = int(point1[0] * image_np.shape[1]), int(point1[1] * image_np.shape[0])
    p2_x, p2_y = int(point2[0] * image_np.shape[1]), int(point2[1] * image_np.shape[0])

    # 座標が画像の範囲内に収まるように修正
    x_min = max(0, min(p1_x, p2_x))
    x_max = min(image_np.shape[1], max(p1_x, p2_x))
    y_min = max(0, min(p1_y, p2_y))
    y_max = min(image_np.shape[0], max(p1_y, p2_y))

    # 指定された矩形領域を黒に設定
    mask[y_min:y_max, x_min:x_max] = 0

    # デバッグ用に元の画像に白い矩形を描画
    cv2.rectangle(debug_image, (x_min, y_min), (x_max, y_max), (255, 255, 255), 2)

    # デバッグ画像を保存
    debug_image_pil = Image.fromarray(debug_image)
    debug_image_pil.save(debug_image_filename)

    # マスク画像を保存
    final_image_pil = Image.fromarray(mask)
    final_image_pil.save(mask_filename)

    return mask_filename


import cv2
import numpy as np
from datetime import datetime
from ultralytics import YOLO
from PIL import Image

def special_process_image_yolo(risk_level, image_path, point1, point2, thresholds=None):
    '''
    point1 = (0.00000000000002, 0.00000000000002)
    point2 = (0.00000000000001, 0.00000000000001)
    '''


    # タイムスタンプを作成
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

    # リスクレベルに基づいた減衰率の計算
    def logistic_decay(risk_level, k=0.1, r0=50):
        return 1 / (1 + np.exp(-k * (risk_level - r0)))

    decay_factor = logistic_decay(risk_level)
    adjusted_thresholds = {key: max(value - decay_factor + 0.8, 0.01) / 2 for key, value in thresholds.items()}

    # YOLOv8モデルをロード
    model = YOLO('/content/drive/MyDrive/learn/1026.pt')  # 適切なモデルパスに変更

    # 画像の読み込みとRGB変換
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_np = np.array(image_rgb, dtype=np.uint8)

    # 推論実行
    results = model(image_rgb)

    # 初期化したマスク画像
    mask = np.zeros((image_np.shape[0], image_np.shape[1]), dtype=np.uint8)

    # 各検出結果に基づきマスク作成
    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])  # ボックスの座標
        confidence = box.conf[0]
        class_id = box.cls[0]
        object_type = model.names[int(class_id)]

        # クラス名に基づいたしきい値
        threshold = adjusted_thresholds.get(object_type, 0.5)

        if confidence >= threshold:
            mask[y1:y2, x1:x2] = 255  # ボックス領域を白に設定

    # 絶対座標に変換した点の範囲を黒に設定
    p1_x, p1_y = int(point1[0] * image_np.shape[1]), int(point1[1] * image_np.shape[0])
    p2_x, p2_y = int(point2[0] * image_np.shape[1]), int(point2[1] * image_np.shape[0])
    x_min, y_min = max(0, min(p1_x, p2_x)), max(0, min(p1_y, p2_y))
    x_max, y_max = min(image_np.shape[1], max(p1_x, p2_x)), min(image_np.shape[0], max(p1_y, p2_y))
    mask[y_min:y_max, x_min:x_max] = 0  # 範囲を黒に設定

    # デバッグ用に白い長方形を描画
    debug_image = image_np.copy()
    cv2.rectangle(debug_image, (x_min, y_min), (x_max, y_max), (255, 255, 255), 2)

    # デバッグ画像とマスク画像を保存
    debug_image_pil = Image.fromarray(debug_image)
    debug_image_pil.save(f"/content/debug_image_with_rectangle_{timestamp}.jpg")

    mask_image_pil = Image.fromarray(mask)
    mask_image_pil.save(f"/content/final_mask_{timestamp}.jpg")

    return f"/content/final_mask_{timestamp}.jpg"


#この下のコードは普通タイプのコード

def process_image(risk_level, image_path, thresholds=None):
    if thresholds is None:
        thresholds = {}

    # Generate the threshold adjustments based on risk level
    def logistic_decay(risk_level, k=0.1, r0=50):
        return 1 / (1 + np.exp(-k * (risk_level - r0)))

    threshold_results = {key: [] for key in thresholds.keys()}
    decay_factor = logistic_decay(risk_level)

    # Adjust thresholds for the given risk level
    adjusted_thresholds = {key: max(value - decay_factor + 0.8, 0.01) / 2 for key, value in thresholds.items()}
    print(adjusted_thresholds)
    detection_model = load_model(CONFIG_PATH, WEIGHTS_PATH)
    risk_level = int(risk_level)
    IMAGE_PATH = image_path
    LABELS = decide_to_object(risk_level)
    image_source, image = load_image(IMAGE_PATH)

    all_boxes = []
    all_logits = []

    for label in LABELS:
        boxes, logits, _ = predict(
            model=detection_model,
            image=image,
            caption=label,
            box_threshold=0.3,
            text_threshold=0.3
        )
        all_boxes.extend(boxes)
        all_logits.extend(logits)

    # Convert PyTorch tensor to NumPy array and ensure it's uint8
    image_np = image.permute(1, 2, 0).cpu().numpy()
    image_np = np.clip(image_np * 255, 0, 255).astype(np.uint8)  # Ensure values are in [0, 255]

    # Initialize mask as black
    mask = np.zeros((image_np.shape[0], image_np.shape[1]), dtype=np.uint8)

    for i, box in enumerate(all_boxes):
        x, y, w, h = box
        confidence = all_logits[i]  # Get the confidence score
        object_type = LABELS[i % len(LABELS)]  # Determine the object type

        # Use the threshold specific to the object type, default to the adjusted threshold if not provided
        threshold = adjusted_thresholds.get(object_type, 0.5)

        if confidence >= threshold:
            # Adjust the box coordinates, convert to integers
            x1 = int((x - w / 2) * image_np.shape[1])
            y1 = int((y - h / 2) * image_np.shape[0])
            x2 = int((x + w / 2) * image_np.shape[1])
            y2 = int((y + h / 2) * image_np.shape[0])

            # Ensure coordinates are within the image dimensions
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(image_np.shape[1], x2), min(image_np.shape[0], y2)

            mask = create_mask(image_np, x1, y1, x2, y2) | mask

    # Save the mask as an image
    final_image_pil = Image.fromarray(mask)
    final_image_pil.save("/content/final_mask.jpg")

    return "/content/final_mask.jpg"



def convert_image_format(input_path, output_format="png"):
    """
    画像をJPGからPNGまたはPNGからJPGに変換する関数。

    Parameters:
    - input_path: 変換したい元画像のパス
    - output_format: 出力形式 ("png" または "jpg" を指定、デフォルトは "png")

    Returns:
    - output_path: 変換された画像の出力パス
    """
    # サポートされているフォーマットかを確認
    if output_format not in ["png", "jpg", "jpeg"]:
        raise ValueError("サポートされている出力形式は 'png' または 'jpg' です。")

    # 画像の読み込み
    image = cv2.imread(input_path)
    if image is None:
        raise ValueError(f"画像が見つかりません: {input_path}")

    # 出力パスの生成
    base_name = os.path.splitext(os.path.basename(input_path))[0]
    output_path = f"{base_name}.{output_format}"

    # 画像の保存
    if output_format == "png":
        cv2.imwrite(output_path, image, [cv2.IMWRITE_PNG_COMPRESSION, 9])  # PNG形式で最高圧縮率
    else:
        cv2.imwrite(output_path, image, [cv2.IMWRITE_JPEG_QUALITY, 90])    # JPG形式で高画質

    return output_path


'''
#下は反比例
#thresholdにプラスでlower limitsの値も辞書にしておくこと
def process_image(risk_level, image_path, thresholds=None, lower_limits=None):
    if thresholds is None:
        thresholds = {}
    if lower_limits is None:
        lower_limits = {}

    #閾値調整関数（反比例）
    def adjust_threshold(initial_threshold, risk_level, lower_limit):
        adjusted_value = max(initial_threshold / (1 + risk_level / 20), lower_limit)
        return adjusted_value  #下限を適用

    threshold_results = {key: [] for key in thresholds.keys()}

    # Adjust thresholds for the given risk level
    adjusted_thresholds = {
        key: adjust_threshold(value, risk_level, lower_limits.get(key, 0.01))
        for key, value in thresholds.items()
    }
    print(adjusted_thresholds)

    #したは元コード
    detection_model = load_model(CONFIG_PATH, WEIGHTS_PATH)
    risk_level = int(risk_level)
    IMAGE_PATH = image_path
    LABELS = decide_to_object(risk_level)
    image_source, image = load_image(IMAGE_PATH)

    all_boxes = []
    all_logits = []

    for label in LABELS:
        boxes, logits, _ = predict(
            model=detection_model,
            image=image,
            caption=label,
            box_threshold=0.3,
            text_threshold=0.3
        )
        all_boxes.extend(boxes)
        all_logits.extend(logits)

    #Convert PyTorch tensor to NumPy array and ensure it's uint8
    image_np = image.permute(1, 2, 0).cpu().numpy()
    image_np = np.clip(image_np * 255, 0, 255).astype(np.uint8)  #値は[0, 255]のあいだ

    #Initialize mask as black
    mask = np.zeros((image_np.shape[0], image_np.shape[1]), dtype=np.uint8)

    for i, box in enumerate(all_boxes):
        x, y, w, h = box
        confidence = all_logits[i]  #Get the confidence score
        object_type = LABELS[i % len(LABELS)]  #Determine the object type

        #Use the threshold specific to the object type, default to the adjusted threshold if not provided
        threshold = adjusted_thresholds.get(object_type, 0.5)

        if confidence >= threshold:
            # Adjust the box coordinates, convert to integers
            x1 = int((x - w / 2) * image_np.shape[1])
            y1 = int((y - h / 2) * image_np.shape[0])
            x2 = int((x + w / 2) * image_np.shape[1])
            y2 = int((y + h / 2) * image_np.shape[0])

            # Ensure coordinates are within the image dimensions
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(image_np.shape[1], x2), min(image_np.shape[0], y2)

            mask = create_mask(image_np, x1, y1, x2, y2) | mask

    #Save the mask as an image
    final_image_pil = Image.fromarray(mask)
    final_image_pil.save("/content/final_mask.jpg")

    return "/content/final_mask.jpg"

'''


#この下は、openCV
def inpaint_image_with_mask(image_path, mask_path, output_path, inpaint_radius=5, inpaint_method=cv2.INPAINT_TELEA):
    """
    マスク画像を使用して元画像のインペイントを行う関数。

    Parameters:
    - image_path: 元画像のパス
    - mask_path: マスク画像のパス（修復したい領域が白、その他が黒）
    - output_path: インペイント結果の出力パス
    - inpaint_radius: インペイントの半径（デフォルトは5）
    - inpaint_method: インペイントのアルゴリズム（デフォルトはcv2.INPAINT_TELEA）

    Returns:
    - inpainted_image: インペイントされた画像
    """
    # 画像とマスクを読み込み
    image = cv2.imread(image_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)  # マスクはグレースケールで読み込み

    # マスク画像が正常に読み込めたかチェック
    if image is None:
        raise ValueError(f"元画像が見つかりません: {image_path}")
    if mask is None:
        raise ValueError(f"マスク画像が見つかりません: {mask_path}")

    # マスク画像が元画像と同じサイズでない場合、リサイズ
    if image.shape[:2] != mask.shape[:2]:
        print(f"マスク画像のサイズを元画像に合わせてリサイズします: {mask.shape} -> {image.shape[:2]}")
        mask = cv2.resize(mask, (image.shape[1], image.shape[0]))

    # インペイント処理
    inpainted_image = cv2.inpaint(image, mask, inpaint_radius, inpaint_method)

    # インペイント結果を保存
    cv2.imwrite(output_path, inpainted_image)

    return  output_path


def stamp_image_with_mask(base_image_path, mask_path,output_path,stamp_image_path='/content/drive/MyDrive/main.png'):
    """
    マスク画像を使用して元画像に別の画像を埋め込む関数。

    Parameters:
    - base_image_path: 元画像のパス
    - mask_path: マスク画像のパス（埋め込みたい領域が白、その他が黒）
    - embed_image_path: 埋め込み用画像のパス
    - output_path: 結果の出力パス

    Returns:
    - output_path: 埋め込み処理された画像の出力パス
    """
    # 画像とマスクを読み込み
    base_image = cv2.imread(base_image_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    embed_image = cv2.imread(stamp_image_path)

    # 画像が正常に読み込めたかチェック
    if base_image is None:
        raise ValueError(f"元画像が見つかりません: {base_image_path}")
    if mask is None:
        raise ValueError(f"マスク画像が見つかりません: {mask_path}")
    if embed_image is None:
        raise ValueError(f"埋め込み用画像が見つかりません: {stamp_image_path}")

    # マスク画像と埋め込み画像を元画像と同じサイズにリサイズ
    if base_image.shape[:2] != mask.shape[:2]:
        print(f"マスク画像のサイズを元画像に合わせてリサイズします: {mask.shape} -> {base_image.shape[:2]}")
        mask = cv2.resize(mask, (base_image.shape[1], base_image.shape[0]))
    if base_image.shape[:2] != embed_image.shape[:2]:
        print(f"埋め込み画像のサイズを元画像に合わせてリサイズします: {embed_image.shape[:2]} -> {base_image.shape[:2]}")
        embed_image = cv2.resize(embed_image, (base_image.shape[1], base_image.shape[0]))

    # マスク領域に埋め込み画像を配置
    embedded_image = base_image.copy()
    embedded_image[mask == 255] = embed_image[mask == 255]

    # 結果を保存
    cv2.imwrite(output_path, embedded_image)

    return output_path

#この下はlama-simpleのinpaint
def inpaint_image_with_mask1(img_path, mask_path, output_path, resize_factor=0.5):
    print('lama')
    # 画像とマスクを読み込み
    image = Image.open(img_path)
    mask = Image.open(mask_path).convert('L')  # マスクをグレースケールに変換

    # 画像とマスクのサイズを合わせる
    mask = mask.resize(image.size, Image.NEAREST)
   #マスクのエッジをぼかす (Gaussian Blur)
    blurred_mask = mask.filter(ImageFilter.GaussianBlur(radius=3))  #半径3ピクセルでぼかし
    # SimpleLama インスタンスを作成
    simple_lama = SimpleLama()

    # インペイントの実行
    result = simple_lama(image, mask)

    # 出力画像をリサイズ
    new_size = (int(result.width * resize_factor), int(result.height * resize_factor))
    result = result.resize(new_size, Image.ANTIALIAS)

    # 結果を保存
    result.save(output_path)
    print(f"Inpainted image saved at {output_path}")
    return output_path



# SDパイプラインをグローバルに初期化
pipe = StableDiffusionInpaintPipeline.from_pretrained(
            "stabilityai/stable-diffusion-2-inpainting", torch_dtype=torch.float16
        ).to("cuda")

def load_inpainting_pipeline():
    global pipe
    if pipe is None:
        pipe = StableDiffusionInpaintPipeline.from_pretrained(
            "stabilityai/stable-diffusion-2-inpainting", torch_dtype=torch.float16
        ).to("cuda")
    return pipe

def inpaint_images(image_paths, mask_paths, output_dir="output_images"):
    # パイプラインの読み込み
    load_inpainting_pipeline()

    # 出力ディレクトリの作成
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 並列処理でインペインティングを実行
    with ThreadPoolExecutor() as executor:
        futures = []
        for i, (image_path, mask_path) in enumerate(zip(image_paths, mask_paths)):
            # 画像とマスクを読み込み
            image = cv2.imread(image_path)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            # 画像とマスクの読み込みチェック
            if image is None:
                raise ValueError(f"{image_path} が正しく読み込まれていません。ファイルパスを確認してください。")
            if mask is None:
                raise ValueError(f"{mask_path} が正しく読み込まれていません。ファイルパスを確認してください。")

            # マスクと画像のサイズ確認
            if mask.shape[:2] != image.shape[:2]:
                raise ValueError(f"マスク {mask_path} のサイズが画像 {image_path} のサイズと一致しません。")

            # 画像とマスクをPIL形式に変換
            image_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            mask_pil = Image.fromarray(mask)

            # インペインティング処理の実行を非同期で開始
            futures.append(executor.submit(
                lambda img, m, idx: pipe(
                    prompt="Fill the masked area with contextually appropriate background or scenery that matches the surrounding area.",
                    negative_prompt="text, letters, Exclude people, human figures, arms, and any human body parts from the generated content.",
                    image=img,
                    mask_image=m,
                    num_inference_steps=10,  # ステップ数を増加
                    guidance_scale=7.5  # 高いガイダンススケールを設定
                ).images[0], image_pil, mask_pil, i
            ))

        # 結果を取得して保存
        for i, future in enumerate(futures):
            result = future.result()

            # 結果のリサイズと保存
            result_image = result.resize((image.shape[1], image.shape[0]), Image.LANCZOS)
            output_image_path = os.path.join(output_dir, f"output_{i+1}.png")
            result_image.save(output_image_path)

            # 処理が完了したことを通知
            print(f"{image_paths[i]} と {mask_paths[i]} のペアの処理が完了しました。結果は {output_image_path} に保存されました。")
            return output_dir

#deepfillv2用のコード

# マスクを画像サイズにリサイズする関数
def resize_mask_to_image(mask_image_path, target_image_path):
    """
    マスク画像のサイズを元画像のサイズに合わせる
    """
    # 元画像とマスクを読み込む
    target_image = Image.open(target_image_path)
    mask_image = Image.open(mask_image_path)

    # マスクを元画像のサイズにリサイズ
    resized_mask = mask_image.resize(target_image.size, Image.NEAREST)

    return resized_mask


class StopExecution(Exception):
    def _render_traceback_(self):
        pass

def run_python_script(script_path):
    try:
        # Pythonスクリプトを新しいプロセスとして実行
        result = subprocess.run(['python', script_path], check=True, text=True, capture_output=True)
        # 実行が成功した場合、標準出力を表示
        print("Script output:", result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Error occurred while running script: {e}")
        print("Script stderr:", e.stderr)

def copy_image_to_folder(image_path, save_as_input,save_path=None):
    """
    Copies the given image to the appropriate folder based on the save_as_input flag.

    Parameters:
    image_path (str): The path of the image to be copied.
    save_as_input (int): 0 to save as input image, 1 to save as mask image.
    """
    if not os.path.exists(image_path):
        print(f"Error: The file {image_path} does not exist!")
        raise StopExecution

    # Set the destination folder and file name based on the save_as_input flag
    if save_as_input == 0:
        destination_path = "/content/deepfillv2_colab/input/input_img.png"
        print(f"Saving image as input: {destination_path}")
    elif save_as_input == 1:
        destination_path = "/content/deepfillv2_colab/input/mask.png"
        print(f"Saving image as mask: {destination_path}")
    elif save_as_input==2:
        destination_path = save_path
        print(f"Saving image as mask: {destination_path}")
    else:
        print("Error: Invalid save_as_input value. It must be 0 (input) or 1 (mask).")
        raise StopExecution

    # Copy the image to the destination
    shutil.copy(image_path, destination_path)
    print(f"Image {image_path} copied to {destination_path} successfully!")

#deepfillv2
def inpaint_image_with_mask2(input_path, output_path, point1, point2, risk_level,thresholds):


    input_path=convert_image_format(input_path,'png')#PNGコンバート

    #point設定

    save_as_input=0
    mask_path=special_process_image_yolo(risk_level, input_path, point1, point2, thresholds)
    mask_path=convert_image_format(mask_path,'png')
    resize_mask_if_needed(mask_path, input_path)
    copy_image_to_folder(input_path, save_as_input)

    save_as_input=1#option
    copy_image_to_folder(mask_path, save_as_input)
    os.chdir(f"{HOME}/deepfillv2_colab")#一度ディレクトリを変更
    run_python_script('/content/deepfillv2_colab/inpaint.py')#プロセス用のコードを呼び出す
    #!python inpaint.py
    os.chdir(f"{HOME}/GroundingDINO") #元に戻す


    output_image_path='/content/deepfillv2_colab/output/inpainted_img.png'
    save_as_input=2#option
    resize_mask_if_needed(input_path,output_image_path)
    copy_image_to_folder(output_image_path, save_as_input,output_path)


    return output_path




















app = FastAPI()
# CORSミドルウェアの追加
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # ここを適切なオリジンに設定することもできます
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 保存先のディレクトリを指定
SAVE_DIR = Path("/content/saved_images")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

def save_image(file, filename):
    """画像ファイルを指定ディレクトリに保存"""
    filepath = SAVE_DIR / filename
    with open(filepath, "wb") as buffer:
        shutil.copyfileobj(file, buffer)
    return filepath

@app.post("/create-mask-and-inpaint-opencv")
async def create_mask_and_inpaint_opencv(image: UploadFile = File(...), risk_level: int = Form(...)):
    point1 = (0.00000000000002, 0.00000000000002)
    point2 = (0.00000000000001, 0.00000000000001)
    input_path = save_image(image.file, "input.jpg")
    mask_path =  special_process_image_yolo(risk_level, input_path, point1, point2, thresholds)

    output_path = SAVE_DIR / "output_opencv.jpg"
    # OpenCVでインペイント
    stamp_image_with_mask(input_path, mask_path, output_path)

    return FileResponse(output_path)






@app.post("/create-mask-and-inpaint-simple-lama")
async def create_mask_and_inpaint_simple_lama(image: UploadFile = File(...), risk_level: int = Form(...)):
    input_path = save_image(image.file, "input.jpg")
    point1 = (0.00000000000002, 0.00000000000002)
    point2 = (0.00000000000001, 0.00000000000001)
    mask_path =  special_process_image_yolo(risk_level, input_path, point1, point2, thresholds)
    output_path = SAVE_DIR / "output_simple_lama.jpg"
    # SimpleLamaでインペイント
    inpaint_image_with_mask1(input_path, mask_path, output_path, resize_factor=1)

    return FileResponse(output_path)


#下のendpointは特定領域をマスクしないタイプのもの


#下記はDeepFillv2






@app.post("/create-mask-and-inpaint-deepfillv2")
# エンドポイントの実装
async def deepfillv2_inpaint_endpoint(
    image: UploadFile = File(...),
    risk_level: int = Form(...)
):
    point1 = (0.00000000000002, 0.00000000000002)
    point2 = (0.00000000000001, 0.00000000000001)
    input_path = save_image(image.file, "input.jpg")
     # 現在のタイムスタンプを生成
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # 一意な識別子を生成
    unique_id = uuid.uuid4().hex

    # 出力ファイル名を生成
    output_path = f"/content/output_deepfill_V2_{timestamp}_{unique_id}.jpg"

    inpaint_image_with_mask2(input_path,output_path,point1,point2,risk_level,thresholds=thresholds)
    return FileResponse(output_path)






@app.post("/create-mask-and-inpaint-stable-diffusion")
async def create_mask_and_inpaint_stable_diffusion(image: UploadFile = File(...), risk_level: int = Form(...)):
    input_path = save_image(image.file, "input.jpg")
    point1 = (0.00000000000002, 0.00000000000002)
    point2 = (0.00000000000001, 0.00000000000001)
    mask_path =  special_process_image_yolo(risk_level, input_path, point1, point2, thresholds)

    output_path = SAVE_DIR / "output_stable_diffusion.png"

    # 保存された画像の有効性を確認
    validate_image(input_path)


    # マスクと元画像のサイズが一致しない場合はリサイズ
    resize_mask_if_needed(mask_path, input_path)

    # Stable Diffusionでインペイント
    #inpaint_images([resize_mask_if_needed(mask_path,inpaint_image_with_mask1(input_path, mask_path, output_path, resize_factor=1))], [mask_path], output_dir=SAVE_DIR)
    inpaint_images([resize_mask_if_needed(mask_path, input_path)], [mask_path], output_dir=SAVE_DIR)
    return FileResponse(SAVE_DIR / "output_1.png")

def validate_image(filepath):
    try:
        with Image.open(filepath) as img:
            img.verify()
        print(f"Image at {filepath} is valid.")
    except Exception as e:
        print(f"Invalid image file at {filepath}: {e}")
        raise ValueError(f"Invalid image file: {e}")

def resize_mask_if_needed(mask_path, input_path):
    with Image.open(input_path) as input_image:
        with Image.open(mask_path) as mask_image:
            if input_image.size != mask_image.size:
                print(f"Resizing mask from {mask_image.size} to {input_image.size}")
                resized_mask = mask_image.resize(input_image.size)
                resized_mask.save(mask_path)
    return input_path







@app.post("/test_inpaint_with_yolo1")
async def create_mask_and_inpaint_stable_diffusion(image: UploadFile = File(...), risk_level: int = Form(...),option:int=Form(...)):
  input_path = save_image(image.file, "input.jpg")
  mask_path = process_image(risk_level, input_path, thresholds=thresholds) # マスク画像を作成
  point1 = (0.00000000000002, 0.00000000000002)
  point2 = (0.00000000000001, 0.00000000000001)


  if option == 1 :
    mask_path =  special_process_image_yolo(risk_level, input_path, point1, point2, thresholds)
    output_path = SAVE_DIR / "output_opencv.jpg"
    # OpenCVでインペイント
    inpaint_image_with_mask(input_path, mask_path, output_path)
    return FileResponse(output_path)


  elif option == 2 :
    mask_path =  special_process_image_yolo(risk_level, input_path, point1, point2, thresholds)
    output_path = SAVE_DIR / "output_simple_lama.jpg"
    # SimpleLamaでインペイント
    inpaint_image_with_mask1(input_path, mask_path, output_path, resize_factor=1)
    return FileResponse(output_path)


  elif option == 3:
    mask_path =  special_process_image_yolo(risk_level, input_path, point1, point2, thresholds)
    output_path = SAVE_DIR / "output_stable_diffusion.png"
    # 保存された画像の有効性を確認
    validate_image(input_path)
    # マスクと元画像のサイズが一致しない場合はリサイズ
    resize_mask_if_needed(mask_path, input_path)
    # Stable Diffusionでインペイント
    #inpaint_images([resize_mask_if_needed(mask_path,inpaint_image_with_mask1(input_path, mask_path, output_path, resize_factor=1))], [mask_path], output_dir=SAVE_DIR)
    inpaint_images([resize_mask_if_needed(mask_path, input_path)], [mask_path], output_dir=SAVE_DIR)
    return FileResponse(SAVE_DIR / "output_1.png")

  elif option == 4:
    # 現在のタイムスタンプを生成
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # 一意な識別子を生成
    unique_id = uuid.uuid4().hex
    # 出力ファイル名を生成
    output_path = f"/content/output_deepfill_V2_{timestamp}_{unique_id}.jpg"
    inpaint_image_with_mask2(input_path,output_path,point1,point2,risk_level,thresholds=thresholds)
    return FileResponse(output_path)

  else :
    mask_path =  special_process_image_yolo(risk_level, input_path, point1, point2, thresholds)
    output_path = SAVE_DIR / "output_opencv.jpg"
    # OpenCVでインペイント
    stamp_image_with_mask(input_path, mask_path, output_path)
    return FileResponse(output_path)
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse
from PIL import Image, ImageDraw
from datetime import datetime
import uuid
import random
import time

@app.post("/test_inpaint_with_yolo")
async def create_mask_and_inpaint_stable_diffusion(
    image: UploadFile = File(...),
    risk_level: int = Form(...),
    option: int = Form(...),
    mask_percentage: float = Form(...)
):
    # 画像の保存
    input_path = save_image(image.file, "input.jpg")

    # 画像のサイズを取得
    with Image.open(input_path) as img:
        img_width, img_height = img.size

    # マスクのサイズをパーセンテージに基づいて計算
    mask_width = int(img_width * (mask_percentage / 100))
    mask_height = int(img_height * (mask_percentage / 100))
    width = mask_width
    height = mask_height
    image_path = input_path

    # マスクを生成し、四角形の位置を取得
    mask_path, mask_position = create_random_mask(image_path, width, height)

    # オプションごとの処理
    if option == 1:
        output_path = os.path.join(SAVE_DIR, "output_opencv.jpg")
        inpaint_image_with_mask(input_path, mask_path, output_path)
    elif option == 2:
        output_path = os.path.join(SAVE_DIR, "output_simple_lama.jpg")
        inpaint_image_with_mask1(input_path, mask_path, output_path, resize_factor=1)
    elif option == 3:
        output_path = os.path.join(SAVE_DIR, "output_stable_diffusion.png")
        validate_image(input_path)
        resize_mask_if_needed(mask_path, input_path)
        inpaint_images([resize_mask_if_needed(mask_path, input_path)], [mask_path], output_dir=SAVE_DIR)
        output_path = os.path.join(SAVE_DIR, "output_1.png")
    else:
        output_path = os.path.join(SAVE_DIR, "output_opencv.jpg")
        stamp_image_with_mask(input_path, mask_path, output_path)

    # 四角形の枠線を追加
    add_border_to_output(output_path, mask_position)

    return FileResponse(output_path)

def save_image(file, filename):
    path = os.path.join(SAVE_DIR, filename)
    with open(path, "wb") as buffer:
        buffer.write(file.read())
    return path

# マスク画像を生成し、位置を返す
def create_random_mask(image_path, width, height):
    image = Image.open(image_path)
    mask = Image.new("L", image.size, 0)
    draw = ImageDraw.Draw(mask)
    x = random.randint(0, image.size[0] - width)
    y = random.randint(0, image.size[1] - height)
    draw.rectangle([x, y, x + width, y + height], fill=255)
    mask_path = "random_mask.png"
    mask.save(mask_path)
    return mask_path, (x, y, x + width, y + height)

# 出力画像に四角形の枠線を追加
def add_border_to_output(image_path, mask_position):
    with Image.open(image_path) as img:
        draw = ImageDraw.Draw(img)
        draw.rectangle(mask_position, outline="red", width=3)
        img.save(image_path)














# ベクトル化対象のオブジェクトリスト
TEXT_PROMPTS = [
     'text','Name tag', 'License plate', 'Mail', 'Documents', 'QR codes',
        'barcodes', 'Map', 'Digital screens', 'information board',
        'signboard', 'poster', 'sign', 'logo', 'card', 'window', 'mirror',
        'Famous landmark', 'cardboard', 'manhole', 'utility pole'
]
BOX_THRESHOLD = 0.3
TEXT_THRESHOLD = 0.3

# クラスタリング結果をJSONファイルから読み込む関数
def load_sums_from_json(filepath):
    with open(filepath, 'r') as json_file:
        sums = json.load(json_file)
    return sums

# ベクトルデータをJSONファイルから読み込む関数
def load_vectors_from_json(filepath):
    with open(filepath, 'r') as json_file:
        data = json.load(json_file)
    return data

# 新しい画像を分類する関数
def classify_new_image(new_image_vector, sums_data, loaded_vectors, loaded_object_names, k=1):
    cluster_centers = []
    for cluster in sums_data:
        indices = [loaded_object_names.index(obj_name) for obj_name in cluster]
        cluster_vectors = np.array([loaded_vectors[obj_name] for obj_name in cluster])
        cluster_center = np.mean(cluster_vectors, axis=0)
        cluster_centers.append(cluster_center)

    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(cluster_centers, range(len(cluster_centers)))

    new_image_label = knn.predict([new_image_vector])
    return new_image_label[0]

# 画像ベクトル化の処理（例としての関数）
import numpy as np
from ultralytics import YOLO
import cv2

def process_image_vec(image_path):
    # YOLOv8モデルをロード
    model = YOLO('/content/drive/MyDrive/learn/1026.pt')  # モデルのパスを指定してください

    # 初期化
    object_vector = np.zeros(len(TEXT_PROMPTS))

    # 画像の読み込み
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # YOLOで推論を実行
    results = model(image_rgb)

    # 各プロンプトごとに確認
    for i, text_prompt in enumerate(TEXT_PROMPTS):
        prompt_sum = 0  # 各プロンプトに対応するスコアの合計

        for box in results[0].boxes:
            class_id = int(box.cls[0])
            confidence = box.conf[0]
            detected_class = model.names[class_id]

            # 検出クラス名とテキストプロンプトの一致を確認
            if text_prompt.lower() == detected_class.lower():
                prompt_sum += confidence  # クラスが一致した場合、信頼度を加算

        # object_vectorにスコアを格納
        object_vector[i] = prompt_sum

    print(object_vector)
    return object_vector.tolist()






@app.post('/danger_of_value/')
async def solve_image(file: UploadFile = File(...)):
    image_path = "/tmp/temp_image.jpg"

    # アップロードされた画像を保存
    with open(image_path, "wb") as buffer:
        buffer.write(await file.read())
    return process_image_norm(image_path)




def process_image_magnitude(image_path):
    # YOLOv8モデルをロード
    model = YOLO('/content/drive/MyDrive/learn/1026.pt')  # モデルのパスを指定してください

    # 初期化
    object_vector = np.zeros(len(TEXT_PROMPTS))

    # 画像の読み込み
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # YOLOで推論を実行
    results = model(image_rgb)

    # 各プロンプトごとに確認
    for i, text_prompt in enumerate(TEXT_PROMPTS):
        prompt_sum = 0  # 各プロンプトに対応するスコアの合計

        for box in results[0].boxes:
            class_id = int(box.cls[0])
            confidence = box.conf[0]
            detected_class = model.names[class_id]

            # 検出クラス名とテキストプロンプトの一致を確認
            if text_prompt.lower() == detected_class.lower():
                prompt_sum += confidence  # クラスが一致した場合、信頼度を加算

        # object_vectorにスコアを格納
        object_vector[i] = prompt_sum

    # ベクトルの大きさ（ノルム）を計算して返す
    magnitude = np.linalg.norm(object_vector)
    print(magnitude)
    return magnitude

# APIのエンドポイント
@app.post("/classify-image/")
async def classify_image(file: UploadFile = File(...)):
    image_path = "/tmp/temp_image.jpg"

    # アップロードされた画像を保存
    with open(image_path, "wb") as buffer:
        buffer.write(await file.read())

    # 画像をベクトル化
    new_image_vector = process_image_vec(image_path)

    # JSONファイルからデータを読み込む
    json_filepath = "/content/drive/MyDrive/lastsum/output_vectors.json"
    loaded_data = load_vectors_from_json(json_filepath)
    loaded_vectors = {obj_name: np.array(vector) for obj_name, vector in loaded_data.items()}
    loaded_object_names = list(loaded_vectors.keys())

    # 既存のクラスタリング結果を読み込む
    sums_data = load_sums_from_json("/content/drive/MyDrive/lastsum/sums_data.json")

    # 新しい画像がどのクラスタに分類されるかを判定
    new_image_cluster = classify_new_image(new_image_vector, sums_data, loaded_vectors, loaded_object_names)

    return {"danger":dangerarray[int(new_image_cluster + 1)]}#バグったらここを＋にして


@app.post("/create-mask-and-inpaint-simple-lama-special")
async def create_mask_and_inpaint_simple_lama(
    image: UploadFile = File(...),
    risk_level: int = Form(...),
    x1: float = Form(...),
    y1: float = Form(...),
    x2: float = Form(...),
    y2: float = Form(...),
):
    # Extract points from the form data
    point1 = [x1, y1]
    point2 = [x2, y2]

    # Save the input image
    input_path = save_image(image.file, "input.jpg")
    print('1111',point1,point2)
    # Create a mask image (using the new process_image function)
    mask_path = special_process_image_yolo(risk_level, input_path, point1, point2,thresholds=thresholds)

    # Define the output path for the inpainted image
    output_path = "/content/output_simple_lama.jpg"

    # Perform inpainting with SimpleLama
    inpaint_image_with_mask1(input_path, mask_path, output_path, resize_factor=1)

    # Return the resulting image as a response
    return FileResponse(output_path, media_type="image/jpeg", filename="output_simple_lama.jpg")




from PIL import Image

def resize_mask_to_match(image_path, mask_path):
    # オリジナル画像とマスク画像を読み込む
    original_image = Image.open(image_path)
    mask_image = Image.open(mask_path)

    # マスク画像をオリジナル画像のサイズにリサイズ
    resized_mask = mask_image.resize(original_image.size)

    # マスク画像を上書き保存
    resized_mask.save(mask_path)

@app.post("/create-mask-and-inpaint-sum")
async def create_mask_sum(image: UploadFile = File(...), risk_level: int = Form(...),
    x1: float = Form(...),
    y1: float = Form(...),
    x2: float = Form(...),
    y2: float = Form(...),):
    '''
    input_path = save_image(image.file, "/content/saved_images/input.jpg")
    default_x = 0.001
    default_y = 0.001


    point1 = [default_x if math.isnan(x1) else x1, default_y if math.isnan(y1) else y1]

    point2 = [default_x if math.isnan(x2) else x2, default_y if math.isnan(y2) else y2]


    print('1111',point1,point2)
    # Create a mask image (using the new process_image function)
    mask_path = special_process_image(risk_level, input_path, point1, point2,thresholds=thresholds)

   # 現在のタイムスタンプを生成
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # 一意な識別子を生成
    unique_id = uuid.uuid4().hex

    # 出力ファイル名を生成
    output_filename = f"/content/output_simple_lama_{timestamp}_{unique_id}.jpg"

    # Define the output path for the inpainted image
    output_path = output_filename

    # Perform inpainting with SimpleLama
    inpaint_image_with_mask1(input_path, mask_path, output_path, resize_factor=1)

    # Return the resulting image as a response
    return FileResponse(output_path, media_type="image/jpeg", filename=f"output_simple_lama_{timestamp}_{unique_id}.jpg")

    '''
    default_x = 0.001
    default_y = 0.001


    point1 = [default_x if math.isnan(x1) else x1, default_y if math.isnan(y1) else y1]

    point2 = [default_x if math.isnan(x2) else x2, default_y if math.isnan(y2) else y2]


    input_path = save_image(image.file, "input.jpg")
    mask_path = special_process_image(risk_level, input_path, point1, point2,thresholds=thresholds)
    # 現在のタイムスタンプを生成
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # 一意な識別子を生成
    unique_id = uuid.uuid4().hex
    output_path = f"/content/output_simple_lama_{timestamp}_{unique_id}.jpg"

    # OpenCVでインペイント
    inpaint_image_with_mask(input_path, mask_path, output_path)

    return FileResponse(output_path)

    '''

    # 現在のタイムスタンプを生成
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # 一意な識別子を生成
    unique_id = uuid.uuid4().hex

    # ファイル名を生成
    output_filename_opencv = f"/content/output_opencv_{timestamp}_{unique_id}.jpg"
    output_filename_sli = f"/content/output_sli_{timestamp}_{unique_id}.jpg"
    output_filename_deepfill = f"/content/deepfillv2_colab/output/inpainted_img_{timestamp}_{unique_id}.png"

    # special_process_imageの実行
    mask_path = special_process_image(risk_level, input_path, point1, point2, thresholds=thresholds)

    # special_process_imageの後にマスク画像とオリジナル画像のサイズを揃える
    resize_mask_to_match(input_path, mask_path)

    original_image_path = input_path
    mask_image_path = mask_path
    small_threshold = 3.0   # 「すごく小さい」の上限（5%）
    medium_threshold = 8.0  # 「小さい」の上限（20%）
    large_threshold = 15.0   # 「大きい」の上限（50%）

    # マスク画像の解析結果に基づいた処理
    if analyze_mask_combined(original_image_path, mask_image_path, small_threshold, medium_threshold, large_threshold)['size'] == 1:  # サイズ1
        if analyze_mask_combined(original_image_path, mask_image_path, small_threshold, medium_threshold, large_threshold)['brightness'] == 1:
            # めっちゃ小さくて明るいもの
            output_path = output_filename_opencv
            inpaint_image_with_mask(input_path, mask_path, output_path)
        elif analyze_mask_combined(original_image_path, mask_image_path, small_threshold, medium_threshold, large_threshold)['brightness'] == 2:
            # めっちゃ小さくて暗いもの
            output_path = output_filename_sli
            inpaint_image_with_mask1(input_path, mask_path, output_path, resize_factor=1)  # SLI

    elif analyze_mask_combined(original_image_path, mask_image_path, small_threshold, medium_threshold, large_threshold)['size'] == 2:  # サイズ2
        if analyze_mask_combined(original_image_path, mask_image_path, small_threshold, medium_threshold, large_threshold)['brightness'] == 1:
            # 小さくて明るいもの
            input_path = save_image(image.file, f"input_{timestamp}_{unique_id}.jpg")
            save_as_input = 0
            copy_image_to_folder(input_path, save_as_input)
            input_path = process_image(risk_level, input_path, thresholds=thresholds)
            save_as_input = 1  # option
            copy_image_to_folder(input_path, save_as_input)
            os.chdir(f"{HOME}/deepfillv2_colab")  # 一度ディレクトリを変更
            run_python_script('/content/deepfillv2_colab/inpaint.py')  # プロセス用のコードを呼び出す
            os.chdir(f"{HOME}/GroundingDINO")  # 元に戻す
            output_image_path = output_filename_deepfill  # DeepFillv2
        elif analyze_mask_combined(original_image_path, mask_image_path, small_threshold, medium_threshold, large_threshold)['brightness'] == 2:
            # 小さくて暗いもの
            output_path = output_filename_sli
            inpaint_image_with_mask1(input_path, mask_path, output_path, resize_factor=1)  # SLI

    elif analyze_mask_combined(original_image_path, mask_image_path, small_threshold, medium_threshold, large_threshold)['size'] == 3:  # サイズ3
        if analyze_mask_combined(original_image_path, mask_image_path, small_threshold, medium_threshold, large_threshold)['brightness'] == 1:
            # おおきくて明るいもの
            input_path = save_image(image.file, f"input_{timestamp}_{unique_id}.jpg")
            save_as_input = 0
            copy_image_to_folder(input_path, save_as_input)
            input_path = process_image(risk_level, input_path, thresholds=thresholds)
            save_as_input = 1  # option
            copy_image_to_folder(input_path, save_as_input)
            os.chdir(f"{HOME}/deepfillv2_colab")  # 一度ディレクトリを変更
            run_python_script('/content/deepfillv2_colab/inpaint.py')  # プロセス用のコードを呼び出す
            os.chdir(f"{HOME}/GroundingDINO")  # 元に戻す
            output_image_path = output_filename_deepfill  # DeepFillv2
        elif analyze_mask_combined(original_image_path, mask_image_path, small_threshold, medium_threshold, large_threshold)['brightness'] == 2:
            # おおきくて暗いもの
            output_path = output_filename_sli
            inpaint_image_with_mask1(input_path, mask_path, output_path, resize_factor=1)  # SLI

    elif analyze_mask_combined(original_image_path, mask_image_path, small_threshold, medium_threshold, large_threshold)['size'] == 4:  # サイズ4
        if analyze_mask_combined(original_image_path, mask_image_path, small_threshold, medium_threshold, large_threshold)['brightness'] == 1:
            # すごくおおきくて明るいもの
            inpaint_images([input_path], [mask_path], output_dir=SAVE_DIR)  # Diffusion
        elif analyze_mask_combined(original_image_path, mask_image_path, small_threshold, medium_threshold, large_threshold)['brightness'] == 2:
            # すごくおおきくて暗いもの
            output_path = output_filename_sli
            inpaint_image_with_mask1(input_path, mask_path, output_path, resize_factor=1)  # SLI

    # 結果の画像をレスポンスとして返す
    return FileResponse(output_path, media_type="image/jpeg", filename=output_path.split('/')[-1])
'''


@app.get("/", response_class=HTMLResponse)
async def read_root():
    html_content = """
 <!DOCTYPE html>
<html lang="ja">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>画像処理アプリ</title>
    <!-- Bootstrap CSS -->
    <link href="https://stackpath.bootstrapcdn.com/bootstrap/4.5.2/css/bootstrap.min.css" rel="stylesheet">
    <!-- jQuery UI CSS -->
    <link rel="stylesheet" href="https://code.jquery.com/ui/1.12.1/themes/base/jquery-ui.css">
    <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.4/css/all.min.css" rel="stylesheet">
    <style>
        body {
            background-color: #f0f0f5;
            color: #333;
            text-align: center;
            padding: 40px 20px;
        }
        h1 {
            color: #555;
            margin-bottom: 30px;
            font-weight: bold;
        }
        .image-preview, .processed-preview {
            max-width: 100%;
            height: auto;
            border-radius: 10px;
            box-shadow: 0 4px 15px rgba(0, 0, 0, 0.2);
            margin-top: 20px;
        }
        #result {
            margin-top: 40px;
            display: none;
        }
        .slider-container {
            text-align: left;
            margin-top: 20px;
        }
        .slider-label {
            font-size: 1.2rem;
            color: #333;
        }
        #slider {
            margin-top: 10px;
        }
        .btn-primary {
            background-color: #007bff;
            border-color: #007bff;
            font-size: 1.2rem;
            padding: 10px 20px;
            border-radius: 50px;
        }
        .btn-primary:hover {
            background-color: #0056b3;
            border-color: #004085;
        }
        .form-control, .custom-select {
            border-radius: 20px;
            box-shadow: 0 2px 5px rgba(0, 0, 0, 0.1);
        }
        .form-control-file {
            font-size: 1rem;
        }
        .form-group {
            margin-bottom: 25px;
        }
        .btn-success {
            padding: 10px 20px;
            border-radius: 50px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1><i class="fas fa-image"></i> 画像処理アプリ - モザイクとインペイント</h1>

        <div class="form-group">
            <input type="file" id="uploadImage" class="form-control-file" accept="image/*" onchange="previewImage()">
        </div>
        <img id="uploadedImage" class="image-preview" src="#" alt="アップロードされた画像" style="display: none;">

        <div class="form-group mt-4">
            <label for="processingType">処理方法を選択:</label>
            <select id="processingType" class="custom-select">
                <option value="opencv">OpenCVインペイント</option>
                <option value="simple_lama">Simple Lamaインペイント</option>
                <option value="stable_diffusion">Stable Diffusionインペイント</option>
                <option value="deep_fill_v2">DeepFillv2インペイント</option>
            </select>
        </div>

        <div class="slider-container">
            <label for="riskLevel" class="slider-label">リスクレベル (0-100): <span id="riskLevelLabel">50</span></label>
            <div id="slider"></div>
        </div>

        <button class="btn btn-primary mt-4" onclick="processImage()">処理開始</button>

        <div id="result" class="mt-5">
            <h2>処理結果:</h2>
            <img id="processedImage" class="processed-preview" src="" alt="">
            <a id="downloadLink" class="btn btn-success mt-3" href="#" download="processed_image.jpg">処理された画像をダウンロード</a>
        </div>
    </div>

    <!-- jQuery and Bootstrap JS -->
    <script src="https://code.jquery.com/jquery-3.5.1.min.js"></script>
    <script src="https://stackpath.bootstrapcdn.com/bootstrap/4.5.2/js/bootstrap.min.js"></script>
    <!-- jQuery UI -->
    <script src="https://code.jquery.com/ui/1.12.1/jquery-ui.js"></script>

    <script>
        $(function() {
            // スライダーの設定
            $("#slider").slider({
                range: "min",
                value: 50, // 初期値
                min: 0,
                max: 100,
                slide: function(event, ui) {
                    $("#riskLevelLabel").text(ui.value);
                }
            });
        });

        function previewImage() {
            const fileInput = document.getElementById('uploadImage');
            const uploadedImage = document.getElementById('uploadedImage');

            if (fileInput.files && fileInput.files[0]) {
                const reader = new FileReader();
                reader.onload = function (e) {
                    uploadedImage.src = e.target.result;
                    uploadedImage.style.display = 'block';
                };
                reader.readAsDataURL(fileInput.files[0]);
            }
        }

        function processImage() {
            const fileInput = document.getElementById('uploadImage');
            const processingType = document.getElementById('processingType').value;
            const riskLevel = $("#slider").slider("value");  // スライダーから値を取得
            const resultDiv = document.getElementById('result');
            const processedImage = document.getElementById('processedImage');
            const downloadLink = document.getElementById('downloadLink');

            if (fileInput.files.length === 0) {
                alert("画像を選択してください。");
                return;
            }

            const file = fileInput.files[0];
            const formData = new FormData();
            formData.append('image', file);
            formData.append('risk_level', riskLevel);  // リスクレベルを追加

            let apiEndpoint;
            if (processingType === "opencv") {
                apiEndpoint = "/create-mask-and-inpaint-opencv";
            } else if (processingType === "simple_lama") {
                apiEndpoint = "/create-mask-and-inpaint-simple-lama";
            } else if (processingType === "stable_diffusion") {
                apiEndpoint = "/create-mask-and-inpaint-stable-diffusion";
            } else if (processingType=="deep_fill_v2"){
                apiEndpoint = "/create-mask-and-inpaint-deepfillv2";
            }

            const url = "https://wired-kitten-adequately.ngrok-free.app" + apiEndpoint;

            fetch(url, {
                method: 'POST',
                body: formData
            })
            .then(response => {
                if (!response.ok) {
                    throw new Error("Network response was not ok");
                }
                return response.blob();
            })
            .then(blob => {
                const objectURL = URL.createObjectURL(blob);
                processedImage.src = objectURL;
                downloadLink.href = objectURL;
                resultDiv.style.display = "block";
            })
            .catch(error => {
                console.error("画像処理に失敗しました。", error);
                alert("画像処理に失敗しました。");
            });
        }
    </script>
</body>
</html>


    """
    return HTMLResponse(content=html_content)


def run_fastapi():
    if __name__ == "__main__":
        nest_asyncio.apply()
        uvicorn.run(app, host="0.0.0.0", port=8000)

def run_ngrok():
    os.system("ngrok http --domain=wired-kitten-adequately.ngrok-free.app 8000")#固定URLを発行


# スレッドを作成
fastapi_thread = threading.Thread(target=run_fastapi)
ngrok_thread = threading.Thread(target=run_ngrok)

# 二つのスレッドを動かす
fastapi_thread.start()

ngrok_thread.start()
fastapi_thread.join()

ngrok_thread.join()

#リサイズいれる

